# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² mass spectrometry dataset (Homo sapiens) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- DOI: [10.71728/senscience.vq4a-28xa](https://doi.org/10.71728/senscience.vq4a-28xa)
- Schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will download the schema and prepare the dataset for inspection and analysis.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for the FAIR² dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

# Print dataset summary
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Identifier (DOI): {metadata.identifier}")


## 2. Data Overview
Review available record sets, their IDs (`@id`), and the fields within those record sets. For robust referencing, all entities are referenced by their `@id`.

> **Tip**: The `mlcroissant` library exposes record set and field structures using their `@id`. This enables precise and consistent referencing of components within the dataset.

In [ ]:
# List all record sets and their fields

print("Available Record Sets (with @id and name):")
for rs in dataset.record_sets:
    print(f"  Record Set: @id = {rs.id}, name = {getattr(rs, 'name', None)}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      Field: @id = {field.id}, name = {getattr(field, 'name', None)}, dataType = {getattr(field, 'data_type', None)}")
    print()

# Show first record from each record set (if possible)
print("\nPreview one record per record set:")
for rs in dataset.record_sets:
    try:
        records_iter = dataset.records(record_set=rs.id)
        rec = next(records_iter)
        print(f"@id = {rs.id} sample record:")
        print(rec)
    except Exception as e:
        print(f"No records for @id = {rs.id} (or failed to load): {e}")


## 3. Data Extraction
Load data from a specific record set (`@id`-based) into DataFrames for analysis. The main record set typically corresponds to the core protein measurement table.

Let's extract the tabular data for all available record sets.

In [ ]:
# Prepare DataFrames for each record set using their @id

dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for @{record_set_id} with {len(df)} records. Columns: {df.columns.tolist()[:8]}{'...' if len(df.columns) > 8 else ''}")
    except Exception as e:
        print(f"Failed to load records for @{record_set_id}: {e}")

# Show columns for the main table (pick the largest or first as core example)
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in @{main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's explore a numeric field (such as abundance, molecular weight, or peptide count) and apply common EDA steps: filtering, normalizing, and grouping.

- **Choose the relevant field `@id`** from the Data Overview section (e.g., `acc`, `peptide_count`, `abundance`).

In [ ]:
# Example parameters (Use appropriate @id for your use case)
# For demonstration, replace the @id values as needed for your dataset

# Use the main record set
record_set_id = record_set_ids[0]
df = dataframes[record_set_id]
print(f"Columns: {df.columns.tolist()}")

# Pick a numeric field. You may need to adjust this depending on your actual schema.
# For example, common fields could include 'MW' (molecular weight), 'peptide_count', 'abundance', etc.
# Let's try to auto-select a likely numeric field:
import numpy as np

# Try to find a suitable numeric column
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]) and df[col].notnull().sum() > 0:
        numeric_field = col
        break

if not numeric_field:
    # Try to convert a likely column to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].notnull().sum() > 0:
                numeric_field = col
                break
        except Exception:
            continue

print(f"Using numeric field: {numeric_field}")

# Filter records with the numeric field above a threshold
threshold = df[numeric_field].mean() if numeric_field else 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
print(filtered_df[[numeric_field]].head())

# Normalization (z-score)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} (first 5 records):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a string/categorical field if available
group_field = None
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]) and df[col].nunique() < 10:
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped mean {numeric_field} by {group_field} (first 5 groups):")
    print(grouped_df.head())
else:
    print("No suitable group/categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the numeric field and explore relationships. Replace field names if needed depending on the actual field in your dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Histogram of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If a group field exists, boxplot by group
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We loaded the FAIR² human mass spectrometry dataset using `mlcroissant`, identified record sets and their ids, and extracted data for exploration.
- We demonstrated EDA and visual analysis: filtering, normalization, and grouping based on key protein attributes.
- This notebook can now be extended for advanced analysis, such as comparing conditions, discovering outliers, or training machine learning models with the well-documented, semantic Croissant dataset.
